# A foundation neural-network state in Quantax

This example trains **one set of parameters** for several Hamiltonians, then evaluates couplings that were not used in training. It implements the ensemble-SR idea of [Rende et al., Nature Communications 16, 7213 (2025)](https://doi.org/10.1038/s41467-025-62098-x).

Use the separate `foundation-nqs` branch. From its checkout, install into a development environment:

```bash
python -m pip install -e '.[full]'
```

Select that environment's Jupyter kernel. **Restart the kernel before rerunning the geometry/training cell**: Quantax keeps one global Hilbert space. This notebook performs only a small six-site benchmark and submits no cluster jobs.

The supplied Transformer concatenates global couplings to each configuration patch. It uses learned positional embeddings rather than the paper's specialized attention and does not load pretrained weights. The current family interface uses a separately compiled view for each coupling tuple, so this is a prototype for modest grids, not a reproduction of the paper's thousand-system throughput.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
%matplotlib inline
candidates = [Path.cwd(), Path.cwd()/'quantax', *Path.cwd().parents]
REPO = next(path for path in candidates if (path/'quantax/__init__.py').exists())
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO/'examples/foundation_nqs'))
import quantax as qtx
print('Quantax source:', qtx.__file__)
from benchmark import train


## Train at three fields; test two held-out fields

We use Quantax's Pauli-matrix Hamiltonian, H = −Σᵢ σᶻᵢσᶻᵢ₊₁ − h Σᵢ σˣᵢ, with periodic boundaries. Training fields are h=0.6,1.0,1.4; held-out fields are 0.8 and 1.2. The model has real parameters and positive real amplitudes, suitable for this example.

For this tiny system, enumerate all 64 configurations and use their normalized Born probabilities in `Samples.reweight_factor`. This removes sampling noise while testing the same shared SR code used with MCMC. The optimizer sees only Hamiltonians and its own wavefunction, **not exact ground states**. Exact diagonalization is used to assess accuracy.

Each Hamiltonian has its own centered log derivatives and energy residuals. The resulting matrices are weighted and stacked for one shared solve; individual SR updates are never averaged. A C0=1 cap bounds the raw update, which is multiplied by 0.1 here.


The executable benchmark uses a fixed two-term HamiltonianFamily and dynamic batched amplitudes, local energies and Jacobians. The Hamiltonian grid is never dispatched through a Python evaluation loop.

In [ ]:
family, optimizer, history, reference, report = train(steps=120, sites=6)
for h, energy, exact, error in zip(report['evaluated_fields'], report['variational_energies'], report['exact_energies'], report['relative_errors']):
    kind = 'training' if h in report['training_fields'] else 'held out'
    print(f'h={h:.1f} ({kind:8s})  E={energy:.8f}  ED={exact:.8f}  relative error={error:.3e}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
fields = np.array(report['training_fields'])
for i, h in enumerate(fields):
    axes[0].semilogy(np.arange(1, len(history)+1), np.maximum((history[:,i]-reference[i])/abs(reference[i]), 1e-12), label=f'h={h:g}')
axes[0].set(xlabel='Shared training update', ylabel='Relative energy error', title='Training: one shared network')
axes[0].legend(frameon=False)
all_fields = np.array(report['evaluated_fields'])
relative = np.array(report['relative_errors'])
trained = np.isin(all_fields, fields)
axes[1].semilogy(all_fields[trained], relative[trained], 'o', label='Training couplings')
axes[1].semilogy(all_fields[~trained], relative[~trained], 's', label='Held-out interpolation')
axes[1].set(xlabel='h/J', ylabel='Relative energy error', title='Evaluation without fine-tuning')
axes[1].legend(frameon=False)
for ax in axes: ax.grid(alpha=.2)
plt.show()
plt.close(fig)


## Coupling-space fidelity metric

Differentiate the learned wavefunction with respect to h while holding all network weights fixed. The covariance of these log derivatives is the coupling-space QGT; its real part is the fidelity susceptibility in the chosen coupling coordinate. A coupling-dependent normalization or phase drops out.

This measures the learned family, so errors in interpolation can also affect the metric. The small-system curve below is not a determination of a thermodynamic transition.


In [ ]:
import jax
import jax.numpy as jnp
family.symm.basis_make()
spins = qtx.utils.ints_to_array(family.symm.basis.states)
metric_fields = np.linspace(.6, 1.4, 13)
metric = []
for h in metric_fields:
    psi = family.at([float(h)])(spins)
    probability = jax.nn.softmax(2*psi.logabs)
    G = family.coupling_qgt([float(h)], spins, weights=probability)
    metric.append(float(G[0,0].real)/6)
fig, ax = plt.subplots(figsize=(6, 3.5), layout='constrained')
ax.plot(metric_fields, metric, 'o-')
ax.set(xlabel='h/J', ylabel='Learned fidelity metric / N', title='Analytic input derivatives')
ax.grid(alpha=.2)
plt.show()
plt.close(fig)


## Batched persistent MCMC

For larger systems, replace exact enumeration with a FamilyMetropolis sampler. It samples all coupling/chain pairs together with dynamic contexts and shares proposal and forward kernels across the grid. This optional cell is disabled so the default benchmark stays deterministic.

In [ ]:
RUN_MCMC = False
if RUN_MCMC:
    sampler = qtx.sampler.FamilyMetropolis(
        family, optimizer.hamiltonians.couplings,
        samples_per_coupling=128, kind="local_flip",
        thermal_steps=240, sweep_steps=24,
    )
    for step in range(3):
        family.update(.1 * optimizer.get_step(sampler.sweep()))
        print(step, optimizer.energies, optimizer.variances)


## Fine-tune one coupling or save shared weights

`family.at([h])` is a read-only view that stays linked to the family. `family.detach([h])` makes an independent Variational state for ordinary fine-tuning. Save and restore shared model weights and optimizer buffers through the existing Quantax interfaces. An MCMC restart additionally requires each sampler's configurations, proposal settings, the RNG and history; see the [README](foundation_nqs/README.md).


In [ ]:
single_state = family.detach([.9])
single_optimizer = qtx.optimizer.SR(single_state, qtx.operator.Ising(h=.9))
print('Shared parameters:', family.nparams, '; detached parameters:', single_state.nparams)
# family.save('foundation.eqx')
# optimizer.save('optimizer.eqx')


## Starting an anisotropic triangular Hubbard family

Use a **fresh interpreter** before changing the Hilbert space. The helper in `foundation_nqs/triangular_hubbard.py` builds balanced half-filled even L×L tori with the distinct hopping along physical x. It uses physical inputs `(U,t_prime)`, `coupling_scale=(10,1)`, complex amplitudes, a fixed three-term basis and a batched sampler. `build()` returns `(lattice, state, optimizer, sampler, H)`. Thermalize chains before starting training.

## Batched triangular Hubbard grid
Run the following template in a **fresh kernel**, after initializing a periodic spinful triangular lattice and complex default dtype. It shares one three-term operator basis across 400 Hamiltonians. Each sample uses its own conditioned wavefunction; no Python loop dispatches Hamiltonians.
```python
import numpy as np
# Initialize a periodic spinful triangular lattice at balanced half filling first.
H = qtx.operator.Anisohub(
    U=np.linspace(4, 16, 20), t_prime=np.linspace(0.8, 1, 20),
)
model = qtx.model.ConditionalTransformer(
    2, dim=32, heads=4, layers=2, complex_output=True,
    coupling_scale=(10, 1),
)
family = qtx.state.Foundation(model, H.couplings[0])
optimizer = qtx.optimizer.FoundationSR(family, H, C0=0.1)
sampler = qtx.sampler.FamilyMetropolis(
    family, H.couplings, samples_per_coupling=128,
    kind="particle_hop", thermal_steps=100,
)
for iteration in range(100):
    samples = sampler.sweep()  # (400, 128, Nmodes)
    family.update(optimizer.get_step(samples))
    print(optimizer.energies.reshape(H.grid_shape))
```
`H.local_terms(family, samples)` has shape `(400, S, 3)`; `H.combine(terms)` takes the coefficient-weighted sum before estimating variance. The `triangular_hubbard.py` helper constructs the complete half-filled setup.